# 11 · Storage & versioning — Nessie catalog + Apache Iceberg

The companion to notebook `10` (lakeFS). Same *git-for-data* idea, one layer up: where
lakeFS versions **objects**, this notebook versions **tables** — with **Apache Iceberg**
as the table format and **Nessie** as the git-like catalog on top of it.

## Two things, two jobs

**Apache Iceberg — the *table format*.** Iceberg turns a pile of Parquet files in object
storage into a real table with database-grade guarantees:

| Iceberg gives you | what it means |
|-------------------|---------------|
| **snapshots** | every write produces an immutable, addressable version of the whole table — the basis of time-travel |
| **hidden partitioning** | queries never spell out partition columns; Iceberg tracks the partition spec and prunes files for you |
| **schema evolution** | add / drop / rename / reorder columns by id, safely, with no table rewrite — old data still reads |
| **atomic commits** | a write either fully lands or doesn't; readers never see a half-written table |

**Nessie — the *catalog*.** Iceberg needs a catalog to point "this table name" at "this
current metadata file". Nessie is that catalog, and it adds **git semantics over the
*whole* catalog**: `branch`, `tag`, `merge`, `log` — but across **many tables at once**,
not one. A Nessie branch is a consistent view of *every* table as of a catalog commit.

## Three versioning layers, three scopes

The mesh stacks all three; they version at different granularities:

| layer | tool | unit versioned | scope of one "commit" |
|-------|------|----------------|------------------------|
| object store | **lakeFS** | objects (any file) | the whole repo — every dataset at once |
| catalog | **Nessie** | tables | every table in the catalog at once |
| table | **Iceberg** | one table | that single table |

- **Iceberg snapshots** are *per-table* — a new version of one table.
- **Nessie branches** are *catalog-wide* — isolate and version *all* tables together
  (multi-table transactions, an isolated "what-if" catalog you can throw away).
- **lakeFS branches** are *object-store-wide* — isolate *all bytes*, format-agnostic.

> **Where pyiceberg meets Nessie.** We connect with **pyiceberg's `RestCatalog`** against
> **Nessie's Iceberg-REST endpoint** (`/iceberg`, *not* `/api/v2` — that's Trino's Nessie
> path — and *not* the plain `rest` catalog type, which Nessie rejects). One deliberate
> consequence you'll see below: **Nessie keeps table history at the *catalog* level, so it
> hands Iceberg back a single live snapshot per table.** Time-travel therefore runs through
> **Nessie commit hashes**, not Iceberg's own snapshot log — which is exactly the point of
> pairing a git-like catalog with the table format.


## Setup

`pyiceberg` is **not** in the singleuser base image, so we install it here (with the
`s3fs` extra for object-store I/O). `polars` and `pyarrow` already ship in the image.

In [1]:
%pip install -q "pyiceberg[s3fs]"


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**, mirroring the lab's production publisher
(`services/weyland-dagster/weyland_pipeline/iceberg_publish.py`). The committed defaults
are the **in-cluster** service URLs; a validation run overrides the env (e.g. to
`kubectl port-forward` targets) without editing the notebook.

We build a **`RestCatalog`** named `nessie` pointed at Nessie's `/iceberg` endpoint, with
the MinIO S3 warehouse creds.

> **One out-of-cluster wrinkle.** When pyiceberg loads a table, **Nessie vends the S3
> config** (endpoint + creds) it wants the client to use for that table's data files — the
> in-cluster MinIO URL. In the pod that's exactly right. From a port-forward it isn't
> reachable, so we provide a tiny `redirect_io()` helper that rewrites a loaded table's S3
> endpoint to `ICEBERG_S3_ENDPOINT_EFFECTIVE`. In the pod that value equals the endpoint
> Nessie already vends, so the helper is a **no-op**; only an out-of-cluster run sets
> `ICEBERG_S3_ENDPOINT_OVERRIDE` to redirect. Catalog metadata always resolves through
> Nessie regardless.

In [2]:
import os, io, json, time, urllib.request, urllib.parse, urllib.error
import pyarrow as pa
import polars as pl
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.io.fsspec import FsspecFileIO
from pyiceberg.table import StaticTable
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, LongType, StringType

# --- endpoints (committed defaults = in-cluster; env overrides for a port-forward run) ---
NESSIE_ICEBERG_URI = os.environ.get(
    "NESSIE_ICEBERG_URI", "http://nessie.data-mesh.svc.cluster.local:19120/iceberg")
NESSIE_API_URI = os.environ.get(
    "NESSIE_API_URI", "http://nessie.data-mesh.svc.cluster.local:19120/api/v2")
S3_ENDPOINT = os.environ.get("ICEBERG_S3_ENDPOINT", "http://minio.minio.svc.cluster.local:9000")
WAREHOUSE = os.environ.get("ICEBERG_WAREHOUSE", "warehouse")
AK = os.environ["ICEBERG_S3_ACCESS_KEY"]
SK = os.environ["ICEBERG_S3_SECRET_KEY"]

# Data-file S3 endpoint the client should actually reach. In the pod this equals what
# Nessie vends; a port-forward run sets ICEBERG_S3_ENDPOINT_OVERRIDE to redirect.
S3_ENDPOINT_EFFECTIVE = os.environ.get("ICEBERG_S3_ENDPOINT_OVERRIDE", S3_ENDPOINT)

catalog = RestCatalog("nessie", **{
    "uri": NESSIE_ICEBERG_URI,
    "warehouse": WAREHOUSE,
    "s3.endpoint": S3_ENDPOINT,
    "s3.access-key-id": AK,
    "s3.secret-access-key": SK,
    "s3.region": "us-east-1",
    "s3.path-style-access": "true",
})

def redirect_io(tbl):
    """Point a loaded table's data-file I/O at S3_ENDPOINT_EFFECTIVE (keeps Nessie-vended
    creds). No-op in the pod, where that endpoint already matches what Nessie vends."""
    props = dict(tbl.io.properties)
    props["s3.endpoint"] = S3_ENDPOINT_EFFECTIVE
    tbl.io = FsspecFileIO(properties=props)
    return tbl

def s3_io_props():
    """Direct S3 FileIO props (for StaticTable time-travel reads, which bypass Nessie)."""
    return {"s3.endpoint": S3_ENDPOINT_EFFECTIVE, "s3.access-key-id": AK,
            "s3.secret-access-key": SK, "s3.region": "us-east-1", "s3.path-style-access": "true"}

def nessie(method, path, body=None):
    """Minimal Nessie v2 REST client (for the catalog-level branch + history demos)."""
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(NESSIE_API_URI + path, data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read())

print("Nessie iceberg :", NESSIE_ICEBERG_URI)
print("Nessie api     :", NESSIE_API_URI)
print("S3 endpoint    :", S3_ENDPOINT_EFFECTIVE, "(effective)")
print("namespaces     :", [".".join(n) for n in catalog.list_namespaces()])

Nessie iceberg : http://localhost:19120/iceberg
Nessie api     : http://localhost:19120/api/v2
S3 endpoint    : http://localhost:9900 (effective)
namespaces     : ['analytics', 'catalog', 'datasets_health', 'datasets_music', 'dbt', 'eval']


### Idempotent start — clear any leftover scratch

Everything we create lives under the scratch namespace **`nb_demo`** and a scratch Nessie
branch **`nb-demo-scratch`**. If a previous run crashed before cleanup, remove them now so
the notebook always starts clean. **Only these two scratch names are ever touched** — every
existing namespace (`analytics`, `catalog`, `datasets_*`, `dbt`, `eval`) is read-only.

In [3]:
DEMO_NS = "nb_demo"          # scratch namespace — created + fully dropped by this notebook
DEMO_TABLE = f"{DEMO_NS}.tracks"
SCRATCH_BRANCH = "nb-demo-scratch"   # scratch Nessie branch — created + deleted by this notebook

def drop_demo_namespace():
    """Drop nb_demo and everything in it, if present. Refuses any non-scratch name."""
    assert DEMO_NS.startswith("nb_demo"), "refusing to touch a non-scratch namespace"
    if (DEMO_NS,) in catalog.list_namespaces():
        for ident in catalog.list_tables((DEMO_NS,)):
            catalog.drop_table(ident)
        catalog.drop_namespace((DEMO_NS,))
        return f"dropped leftover namespace {DEMO_NS!r}"
    return f"namespace {DEMO_NS!r} absent (clean)"

def drop_scratch_branch():
    """Delete the scratch Nessie branch if it exists. Refuses any non-scratch name."""
    assert SCRATCH_BRANCH.startswith("nb-demo-"), "refusing to delete a non-scratch branch"
    try:
        cur = nessie("GET", f"/trees/{SCRATCH_BRANCH}")["reference"]["hash"]
        nessie("DELETE", f"/trees/{SCRATCH_BRANCH}@{cur}")
        return f"deleted leftover branch {SCRATCH_BRANCH!r}"
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return f"branch {SCRATCH_BRANCH!r} absent (clean)"
        raise

print(drop_demo_namespace())
print(drop_scratch_branch())

namespace 'nb_demo' absent (clean)
branch 'nb-demo-scratch' absent (clean)


## Explore the catalog (read-only)

`list_namespaces()` enumerates the namespaces (Iceberg's schemas); `list_tables(ns)`
lists the tables in one. These are all mesh-owned and **read-only** for this notebook.

In [4]:
rows = []
for ns in catalog.list_namespaces():
    tbls = catalog.list_tables(ns)
    rows.append({"namespace": ".".join(ns), "n_tables": len(tbls),
                 "sample_tables": ", ".join(sorted(".".join(t) for t in tbls)[:4])})
pl.DataFrame(rows)

namespace,n_tables,sample_tables
str,i64,str
"""analytics""",1,"""analytics.trending_artists"""
"""catalog""",1,"""catalog.model_catalog"""
"""datasets_health""",65,"""datasets_health.big_five_big5_…"
"""datasets_music""",28,"""datasets_music.audioset_test, …"
"""dbt""",8,"""dbt.mart_artist_popularity, db…"
"""eval""",1,"""eval.eval_scores"""


## Read an existing table (read-only)

We load a small dbt mart — **`dbt.mart_artist_popularity`** — and inspect it without
touching it. `load_table()` returns a `Table`; from it we read the **schema**, the
**current snapshot id**, the **snapshot history**, and the **partition spec**. (We wrap the
load in `redirect_io()` so a port-forward run can also read the data files.)

In [5]:
art = redirect_io(catalog.load_table(("dbt", "mart_artist_popularity")))

print("schema:")
print(art.schema())
print("current snapshot id :", art.metadata.current_snapshot_id)
print("partition spec      :", art.spec(), "(unpartitioned)" if art.spec().is_unpartitioned() else "")
print("history entries     :", len(art.history()))
print("snapshots           :", len(art.snapshots()))

schema:
table {
  1: artist_name: optional string
  2: mbid: optional string
  3: total_plays: optional long
  4: n_listeners: optional long
  5: musicbrainz_url: optional string
}
current snapshot id : 4008748426418645312
partition spec      : [] (unpartitioned)
history entries     : 1
snapshots           : 1


**Scan to a dataframe.** `table.scan(limit=...).to_arrow()` reads the current
snapshot into a pyarrow Table (Iceberg prunes files for you); we hand it to polars.

In [6]:
scan_arrow = art.scan(limit=5).to_arrow()
print(f"scanned {scan_arrow.num_rows} rows x {scan_arrow.num_columns} cols")
pl.from_arrow(scan_arrow)

scanned 5 rows x 5 cols


artist_name,mbid,total_plays,n_listeners,musicbrainz_url
str,str,i64,i64,str
"""regina spektor""","""fbb375f9-48bb-4635-824e-412027…",3517816,13156,null
"""battles""","""8522b9b6-b295-48d7-9a10-8618fb…",432926,2905,null
"""coconut records""","""eddc0911-21fc-4327-ab90-ccc459…",166370,1021,null
"""bombay bicycle club""","""0ae49abe-d6af-44fa-8ab0-b9ace5…",68519,408,null
"""green day""","""084308bd-1654-436f-ba03-df6697…",6078488,22414,null


**Iceberg metadata tables.** `table.inspect.snapshots()` and `table.inspect.files()`
expose the table's own bookkeeping — the snapshot log and the concrete data files backing
the current snapshot.

Note the snapshot count is **1**. That is the Nessie pairing at work: Nessie keeps this
table's *history* in its own catalog commit log (we read that below), and hands Iceberg a
single live snapshot. The real dbt marts show the same shape.

In [7]:
snaps = art.inspect.snapshots()
print("inspect.snapshots() ->", snaps.column_names)
print(pl.from_arrow(snaps).select(["committed_at", "snapshot_id", "operation"]))

files = art.inspect.files()
print(f"\ninspect.files() -> {files.num_rows} data file(s); columns include:",
      files.column_names[:5])

inspect.snapshots() -> ['committed_at', 'snapshot_id', 'parent_id', 'operation', 'manifest_list', 'summary']
shape: (1, 3)
┌─────────────────────────┬─────────────────────┬───────────┐
│ committed_at            ┆ snapshot_id         ┆ operation │
│ ---                     ┆ ---                 ┆ ---       │
│ datetime[ms]            ┆ i64                 ┆ str       │
╞═════════════════════════╪═════════════════════╪═══════════╡
│ 2026-08-30 10:00:30.668 ┆ 4008748426418645312 ┆ append    │
└─────────────────────────┴─────────────────────┴───────────┘

inspect.files() -> 1 data file(s); columns include: ['content', 'file_path', 'file_format', 'spec_id', 'partition']


## Write demo — in the scratch namespace `nb_demo`

Everything below writes **only** into `nb_demo`, which we created-clean above. We build a
tiny Iceberg table and evolve it across several commits, capturing the **Nessie commit
hash** after each so we can time-travel later.

`create_namespace` + `create_table` register the table in Nessie; `append` adds rows in a
new commit; `overwrite` replaces all rows in another. Each call is one atomic Nessie
commit — and, because Nessie collapses the Iceberg snapshot log, one fresh Iceberg snapshot
id.

In [8]:
catalog.create_namespace((DEMO_NS,))
print("created namespace:", DEMO_NS)

def main_hash():
    return nessie("GET", "/trees/main")["reference"]["hash"]

# v1: create + append 3 rows
rows_v1 = pa.table({"track_id": pa.array([1, 2, 3], pa.int64()),
                    "title": ["aria", "nocturne", "etude"]})
t = redirect_io(catalog.create_table(DEMO_TABLE, schema=rows_v1.schema))
t.append(rows_v1)
HASH_V1 = main_hash()

# v2: append 2 more rows
rows_v2 = pa.table({"track_id": pa.array([4, 5], pa.int64()), "title": ["prelude", "fugue"]})
redirect_io(catalog.load_table(DEMO_TABLE)).append(rows_v2)
HASH_V2 = main_hash()

# v3: overwrite — replace the whole table with a single row
rows_v3 = pa.table({"track_id": pa.array([9], pa.int64()), "title": ["encore"]})
redirect_io(catalog.load_table(DEMO_TABLE)).overwrite(rows_v3)
HASH_V3 = main_hash()

cur = redirect_io(catalog.load_table(DEMO_TABLE))
print("Nessie commit hashes captured:")
print("  after append (3 rows) :", HASH_V1[:12])
print("  after append (5 rows) :", HASH_V2[:12])
print("  after overwrite (1 row):", HASH_V3[:12])
print("current live snapshot id :", cur.metadata.current_snapshot_id)
print("current row count        :", cur.scan().to_arrow().num_rows)

created namespace: nb_demo


Nessie commit hashes captured:
  after append (3 rows) : 09a1b0ec85d2
  after append (5 rows) : 52985e29b2b6
  after overwrite (1 row): cb176fafc776
current live snapshot id : 8305728565029607497
current row count        : 1


### The version log lives in **Nessie**, not Iceberg's snapshot list

Because Nessie hands Iceberg a single live snapshot, the growing history is Nessie's
**catalog commit log**. `GET /trees/main/history?fetch=ALL` returns it; we filter to the
commits that touched `nb_demo.tracks`. Each `PUT` operation is one of our writes above.

In [9]:
hist = nessie("GET", "/trees/main/history?maxRecords=25&fetch=ALL")
mine = []
for e in hist["logEntries"]:
    for op in (e.get("operations") or []):
        if op.get("key", {}).get("elements") == [DEMO_NS, "tracks"]:
            mine.append({"nessie_hash": e["commitMeta"]["hash"][:12],
                         "op": op["type"], "table": ".".join(op["key"]["elements"])})
            break
print(f"{len(mine)} Nessie commits touched {DEMO_TABLE}:")
pl.DataFrame(mine)

16 Nessie commits touched nb_demo.tracks:


nessie_hash,op,table
str,str,str
"""cb176fafc776""","""PUT""","""nb_demo.tracks"""
"""52985e29b2b6""","""PUT""","""nb_demo.tracks"""
"""09a1b0ec85d2""","""PUT""","""nb_demo.tracks"""
"""88e58b7d19ce""","""PUT""","""nb_demo.tracks"""
"""98a1fbdcf140""","""DELETE""","""nb_demo.tracks"""
…,…,…
"""e19f2e7cdf5f""","""PUT""","""nb_demo.tracks"""
"""8e2af4a8d6b0""","""DELETE""","""nb_demo.tracks"""
"""74a017f85faf""","""PUT""","""nb_demo.tracks"""


## Time-travel — read the table as of an earlier Nessie commit

The versioning is catalog-level, so we time-travel by **Nessie hash**. Nessie's Iceberg-REST
endpoint serves a table's metadata *as of any ref* via a hash-qualified prefix
(`main@<hash>|warehouse`). We fetch the metadata location at each captured hash and open it
with pyiceberg's **`StaticTable`** — a read-only view of an exact metadata file — proving we
recover the older state (and, next section, the older schema).

In [10]:
def metadata_location_at(ref_expr):
    """Metadata-file location for nb_demo.tracks as of a Nessie ref (e.g. 'main@<hash>')."""
    prefix = urllib.parse.quote(f"{ref_expr}|{WAREHOUSE}", safe="")
    url = f"{NESSIE_ICEBERG_URI}/v1/{prefix}/namespaces/{DEMO_NS}/tables/tracks"
    with urllib.request.urlopen(url) as r:
        return json.loads(r.read())["metadata-location"]

def rows_at(ref_expr):
    loc = metadata_location_at(ref_expr)
    st = StaticTable.from_metadata(loc, properties=s3_io_props())
    return st.scan().to_arrow()

for label, h in [("v1 append", HASH_V1), ("v2 append", HASH_V2), ("v3 overwrite", HASH_V3)]:
    arr = rows_at(f"main@{h}")
    print(f"{label:14s} @ {h[:12]} -> {arr.num_rows} row(s): {arr.column('title').to_pylist()}")

print("\nlive (main)   ->", rows_at("main").num_rows, "row(s)")

v1 append      @ 09a1b0ec85d2 -> 3 row(s): ['aria', 'nocturne', 'etude']
v2 append      @ 52985e29b2b6 -> 5 row(s): ['prelude', 'fugue', 'aria', 'nocturne', 'etude']


v3 overwrite   @ cb176fafc776 -> 1 row(s): ['encore']



live (main)   -> 1 row(s)


## Schema evolution — add a column, old snapshots keep the old schema

Iceberg evolves schemas by column **id**, so it never rewrites data. `update_schema()` is a
transaction; here we add a `plays` column. Existing rows read back with `plays = null`, and —
crucially — **reading an earlier Nessie hash still sees the old schema**, because that
metadata predates the change.

In [11]:
before = [f.name for f in redirect_io(catalog.load_table(DEMO_TABLE)).schema().fields]

with redirect_io(catalog.load_table(DEMO_TABLE)).update_schema() as us:
    us.add_column("plays", LongType(), doc="lifetime play count")
HASH_V4 = main_hash()

evolved = redirect_io(catalog.load_table(DEMO_TABLE))
after = [f.name for f in evolved.schema().fields]
print("schema before add_column :", before)
print("schema after  add_column :", after)

# current data reads back with the new (nullable) column present
print("\ncurrent rows (new schema):")
print(pl.from_arrow(evolved.scan().to_arrow()))

# an older Nessie hash still reads the OLD schema — no 'plays' column
old_arr = rows_at(f"main@{HASH_V1}")
print("\ncolumns @ v1 hash (pre-evolution):", old_arr.column_names)
assert "plays" not in old_arr.column_names, "old snapshot must not see the new column"
print("confirmed: the pre-evolution snapshot has no 'plays' column")

schema before add_column : ['track_id', 'title']
schema after  add_column : ['track_id', 'title', 'plays']

current rows (new schema):
shape: (1, 3)
┌──────────┬────────┬───────┐
│ track_id ┆ title  ┆ plays │
│ ---      ┆ ---    ┆ ---   │
│ i64      ┆ str    ┆ i64   │
╞══════════╪════════╪═══════╡
│ 9        ┆ encore ┆ null  │
└──────────┴────────┴───────┘



columns @ v1 hash (pre-evolution): ['track_id', 'title']
confirmed: the pre-evolution snapshot has no 'plays' column


## Nessie catalog branching (git-like, cross-table)

A **Nessie branch** is a git branch over the *entire catalog* — an isolated view of every
table at once. You branch off `main`, do isolated multi-table work, and either merge back
or throw it away — the catalog-wide analogue of a lakeFS branch.

pyiceberg-against-Nessie here operates on `main` (Nessie forces the ref via its REST config),
so we demonstrate branching at the **Nessie v2 API** level: list references, **create** a
scratch branch off `main`, confirm it, then **delete** it.

In [12]:
# list references (branches + tags)
refs = nessie("GET", "/trees")["references"]
print("references:")
for r in refs[:8]:
    print(f"  {r['type']:6s} {r['name']:20s} @ {r['hash'][:12]}")

# create a scratch branch off main:  POST /trees?name=&type=BRANCH  body = source reference
src = nessie("GET", "/trees/main")["reference"]
created = nessie("POST", f"/trees?name={SCRATCH_BRANCH}&type=BRANCH",
                 {"type": "BRANCH", "name": "main", "hash": src["hash"]})
print(f"\ncreated branch {created['reference']['name']!r} @ {created['reference']['hash'][:12]}")

names = [r["name"] for r in nessie("GET", "/trees")["references"]]
print("scratch branch present in catalog:", SCRATCH_BRANCH in names)

# delete it:  DELETE /trees/<name>@<hash>
h = next(r["hash"] for r in nessie("GET", "/trees")["references"] if r["name"] == SCRATCH_BRANCH)
nessie("DELETE", f"/trees/{SCRATCH_BRANCH}@{h}")
gone = SCRATCH_BRANCH not in [r["name"] for r in nessie("GET", "/trees")["references"]]
print("scratch branch deleted:", gone)

references:
  BRANCH main                 @ 38a71d88c7a7



created branch 'nb-demo-scratch' @ 38a71d88c7a7
scratch branch present in catalog: True
scratch branch deleted: True


## Cleanup — leave the catalog exactly as we found it

Drop every table we created in `nb_demo`, drop the `nb_demo` namespace, and delete any
scratch Nessie branch that lingers. This runs even if a step above failed, then **asserts**
`nb_demo` is gone and no `nb-demo-*` branch remains — so the notebook never leaves residue.

In [13]:
report = {}
report["branch"] = drop_scratch_branch()   # idempotent — also covers the delete above
report["namespace"] = drop_demo_namespace()

remaining_ns = [".".join(n) for n in catalog.list_namespaces() if n[0].startswith("nb_demo")]
remaining_br = [r["name"] for r in nessie("GET", "/trees")["references"]
                if r["name"].startswith("nb-demo-")]
print("cleanup:", report)
print("scratch namespaces remaining:", remaining_ns)
print("scratch branches remaining  :", remaining_br)
assert remaining_ns == [], f"scratch namespace left behind: {remaining_ns}"
assert remaining_br == [], f"scratch branch left behind: {remaining_br}"
print("\nOK — catalog left exactly as found (only mesh-owned namespaces + main remain)")

cleanup: {'branch': "branch 'nb-demo-scratch' absent (clean)", 'namespace': "dropped leftover namespace 'nb_demo'"}
scratch namespaces remaining: []
scratch branches remaining  : []

OK — catalog left exactly as found (only mesh-owned namespaces + main remain)


## When to reach for each layer

**Use Iceberg tables (catalogued by Nessie) when you have *analytics tables*:**
- **Time-travel & reproducibility** — pin a query, a training set, or a report to an exact
  version and re-read the table as it was (here: an earlier Nessie hash via `StaticTable`).
- **Schema evolution** — add / drop / rename columns by id with no rewrite; old data and old
  versions still read (the `plays` column above).
- **Multi-table transactions & isolation** — a **Nessie branch** gives a consistent, isolated
  view of *many* tables at once: ingest onto a branch, validate, then merge the whole set
  atomically — catalog-wide CI/CD for tables.

**Use lakeFS (notebook `10`) when you need git over *raw objects*** — versioning bytes of any
format (Parquet, Arrow, Lance, images) across the whole lake, format-agnostic, before or
below the table layer.

**Use the plain formats (notebooks `01`–`04`) when you just need a file** — Parquet / Arrow /
Avro / Lance with no catalog and no versioning: a one-off extract, an interchange file, a
scratch artifact.

**How they compose in the mesh:** lakeFS versions the *objects* underneath; Iceberg turns the
curated objects into *tables* with snapshots and evolving schemas; Nessie gives those tables a
*git-like catalog* so whole sets of them branch, version, and merge together. Object layer,
table layer, catalog layer — three scopes of "undo" for the data platform.
